# Stanford RNA 3D Folding Part 2 - Submission Notebook

This notebook generates 3D structure predictions for RNA sequences.

**Competition**: [Stanford RNA 3D Folding Part 2](https://www.kaggle.com/competitions/stanford-rna-3d-folding-2)

**Task**: Predict 3D C1' atom coordinates for RNA molecules from sequence.

**Metric**: TM-score (best of 5 predictions per target)

In [ ]:
import os
import sys
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# Check environment
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Model Definition

Self-contained model code for Kaggle notebook submission.

In [ ]:
# Nucleotide vocabulary
NUC_VOCAB = {'<pad>': 0, 'A': 1, 'C': 2, 'G': 3, 'U': 4, '<unk>': 5}
VOCAB_SIZE = len(NUC_VOCAB)

def encode_sequence(seq):
    return [NUC_VOCAB.get(c.upper(), NUC_VOCAB['<unk>']) for c in seq]


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class PairwiseModule(nn.Module):
    def __init__(self, d_model, d_pair=64):
        super().__init__()
        self.proj_left = nn.Linear(d_model, d_pair)
        self.proj_right = nn.Linear(d_model, d_pair)
        self.pair_norm = nn.LayerNorm(d_pair)
        self.pair_mlp = nn.Sequential(
            nn.Linear(d_pair, d_pair * 2), nn.GELU(), nn.Linear(d_pair * 2, d_pair)
        )
        self.pair_to_bias = nn.Linear(d_pair, 1)

    def forward(self, single_repr, mask):
        left = self.proj_left(single_repr)
        right = self.proj_right(single_repr)
        pair_repr = torch.einsum('bid,bjd->bijd', left, right)
        pair_repr = self.pair_norm(pair_repr)
        pair_repr = pair_repr + self.pair_mlp(pair_repr)
        pair_mask = mask.unsqueeze(-1) * mask.unsqueeze(-2)
        pair_repr = pair_repr * pair_mask.unsqueeze(-1)
        attn_bias = self.pair_to_bias(pair_repr).squeeze(-1)
        return pair_repr, attn_bias


class StructureAwareAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head ** -0.5
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask, attn_bias=None):
        B, L, D = x.shape
        h = self.n_heads
        q = self.q_proj(x).view(B, L, h, -1).transpose(1, 2)
        k = self.k_proj(x).view(B, L, h, -1).transpose(1, 2)
        v = self.v_proj(x).view(B, L, h, -1).transpose(1, 2)
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if attn_bias is not None:
            attn = attn + attn_bias.unsqueeze(1)
        mask_2d = mask.unsqueeze(1).unsqueeze(2)
        attn = attn.masked_fill(~mask_2d, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, L, D)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = StructureAwareAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, mask, attn_bias=None):
        x = x + self.attn(self.norm1(x), mask, attn_bias)
        x = x + self.ffn(self.norm2(x))
        return x


class StructureModule(nn.Module):
    def __init__(self, d_model, num_predictions=5):
        super().__init__()
        self.num_predictions = num_predictions
        self.coord_heads = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model), nn.GELU(),
                nn.Linear(d_model, d_model // 2), nn.GELU(),
                nn.Linear(d_model // 2, 3),
            ) for _ in range(num_predictions)
        ])
        self.confidence_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Linear(d_model // 2, num_predictions), nn.Sigmoid(),
        )

    def forward(self, single_repr):
        coords = torch.stack([head(single_repr) for head in self.coord_heads], dim=2)
        confidence = self.confidence_head(single_repr)
        return {'coords': coords, 'confidence': confidence}


class RNAFoldModel(nn.Module):
    def __init__(self, d_model=256, n_heads=8, n_layers=8, d_ff=1024,
                 dropout=0.0, num_predictions=5, max_seq_len=512):
        super().__init__()
        self.d_model = d_model
        self.num_predictions = num_predictions
        self.token_emb = nn.Embedding(VOCAB_SIZE, d_model, padding_idx=0)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len=max_seq_len + 100)
        self.pairwise = PairwiseModule(d_model, d_pair=64)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.structure_module = StructureModule(d_model, num_predictions)

    def forward(self, tokens, mask):
        x = self.token_emb(tokens)
        x = self.pos_enc(x)
        _, attn_bias = self.pairwise(x, mask)
        for layer in self.layers:
            x = x * mask.unsqueeze(-1).float()
            x = layer(x, mask, attn_bias)
        return self.structure_module(x)

print('Model defined successfully.')

## 2. Load Model and Data

In [ ]:
# Configuration
MODEL_CONFIG = {
    'd_model': 256,
    'n_heads': 8,
    'n_layers': 8,
    'd_ff': 1024,
    'dropout': 0.0,
    'num_predictions': 5,
    'max_seq_len': 512,
}

# Auto-detect environment: Kaggle vs local
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    INPUT_DIR = '/kaggle/input/stanford-rna-3d-folding-2'
    MODEL_DIR = '/kaggle/input/rna-fold-model'  # Your uploaded model weights
    OUTPUT_DIR = '/kaggle/working'
else:
    # Local development paths
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if 'notebooks' in os.getcwd() else os.getcwd()
    INPUT_DIR = os.path.join(PROJECT_ROOT, 'data')
    MODEL_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
    OUTPUT_DIR = PROJECT_ROOT

print(f'Environment: {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Input dir: {INPUT_DIR}')
print(f'Model dir: {MODEL_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

# Load test sequences (or create sample data for local testing)
test_csv_path = os.path.join(INPUT_DIR, 'test_sequences.csv')

if os.path.exists(test_csv_path):
    test_df = pd.read_csv(test_csv_path)
    print(f'\nLoaded test_sequences.csv: {len(test_df)} targets')
else:
    print(f'\ntest_sequences.csv not found at {test_csv_path}')
    print('Creating sample test data for local development...')
    # Sample RNA sequences for testing the pipeline
    sample_data = {
        'target_id': ['SAMPLE_001', 'SAMPLE_002', 'SAMPLE_003'],
        'sequence': [
            'GGGCGAUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGGUCCUGUGUUCGAUCCACAGAAUUCGCACCA',  # tRNA-like
            'GGUCCGAGCAGAAGACGGCUACCCAUUCCGAUUGAGUCCUAGAAAGCUUCUUCUUUAAUUUU',  # short hairpin
            'GCGACCGGGGCUGGCUUGGUAAUGGUACUCCCCUGUCACGGGAGAGAAUGUGGGUUCAAAUCCCAUCGGUCGCGCCA',  # another tRNA
        ],
    }
    test_df = pd.DataFrame(sample_data)
    os.makedirs(INPUT_DIR, exist_ok=True)
    test_df.to_csv(test_csv_path, index=False)
    print(f'Sample data saved to {test_csv_path}')

print(f'Test targets: {len(test_df)}')
print(f'Columns: {list(test_df.columns)}')
print(f'Sequence lengths: {test_df["sequence"].str.len().tolist()}')
print(test_df.head())

In [ ]:
# Load model
model = RNAFoldModel(**MODEL_CONFIG).to(device)

# Load weights if available
ckpt_path = os.path.join(MODEL_DIR, 'best_model.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print('Loaded trained model weights.')
else:
    print('WARNING: No checkpoint found. Using random weights.')

model.eval()
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

## 3. Generate Predictions

In [ ]:
@torch.no_grad()
def predict_structure(model, sequence, max_seq_len, device):
    """Predict 5 3D structures for an RNA sequence."""
    tokens = encode_sequence(sequence)
    seq_len = len(tokens)

    if seq_len <= max_seq_len:
        padded = tokens + [0] * (max_seq_len - seq_len)
        mask = [1] * seq_len + [0] * (max_seq_len - seq_len)
        tokens_t = torch.tensor([padded], dtype=torch.long, device=device)
        mask_t = torch.tensor([mask], dtype=torch.bool, device=device)
        pred = model(tokens_t, mask_t)
        coords = pred['coords'][0, :seq_len].cpu().numpy()
    else:
        # Sliding window for long sequences
        window = max_seq_len
        stride = max_seq_len // 2
        coords = np.zeros((seq_len, 5, 3), dtype=np.float32)
        weights = np.zeros((seq_len, 1, 1), dtype=np.float32)
        for start in range(0, seq_len, stride):
            end = min(start + window, seq_len)
            chunk = tokens[start:end]
            chunk_len = len(chunk)
            padded = chunk + [0] * (window - chunk_len)
            mask = [1] * chunk_len + [0] * (window - chunk_len)
            tokens_t = torch.tensor([padded], dtype=torch.long, device=device)
            mask_t = torch.tensor([mask], dtype=torch.bool, device=device)
            pred = model(tokens_t, mask_t)
            coords[start:end] += pred['coords'][0, :chunk_len].cpu().numpy()
            weights[start:end] += 1.0
            if end >= seq_len:
                break
        coords = coords / np.maximum(weights, 1e-8)
    return coords  # (L, 5, 3)


# Generate submission
rows = []
max_seq_len = MODEL_CONFIG['max_seq_len']

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Predicting'):
    target_id = row['target_id']
    sequence = row['sequence']
    coords = predict_structure(model, sequence, max_seq_len, device)

    for resid, nuc in enumerate(sequence):
        entry = {
            'ID': f'{target_id}_{resid + 1}',
            'resname': nuc.upper(),
            'resid': resid + 1,
        }
        for p in range(5):
            entry[f'x_{p+1}'] = round(float(coords[resid, p, 0]), 3)
            entry[f'y_{p+1}'] = round(float(coords[resid, p, 1]), 3)
            entry[f'z_{p+1}'] = round(float(coords[resid, p, 2]), 3)
        rows.append(entry)

submission = pd.DataFrame(rows)
print(f'\nSubmission shape: {submission.shape}')
print(submission.head(10))

## 4. Save Submission

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(output_path, index=False)
print(f'Submission saved to {output_path}')
print(f'Total rows: {len(submission)}')
print(f'Unique targets: {submission["ID"].str.rsplit("_", n=1).str[0].nunique()}')

# Sanity checks
coord_cols = [c for c in submission.columns if c.startswith(('x_', 'y_', 'z_'))]
print(f'\nCoordinate statistics:')
print(submission[coord_cols].describe())